<a href="https://colab.research.google.com/github/frasercrichton/ai-dde-hackthon/blob/feature%2Fleiden-guidelines-doc/team-red/notebooks/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI DDE Hackathon

Update the cell below with your Huggingface token (see: https://huggingface.co/docs/hub/en/security-tokens) and ensure you have permssion to use the LLama 3 Model (https://huggingface.co/meta-llama/Llama-3.1-8B).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git triton


from google.colab import userdata
userdata.get('GITHUB_TOKEN')

!git clone https://{GITHUB_TOKEN}@github.com/frasercrichton/ai-dde-hackthon.git
%cd ai-dde-hackthon
! git checkout feature/leiden-guidelines-doc
%cd team-red
! ls

# Install Poetry
# !curl -sSL https://install.python-poetry.org | python3 -

# # Add Poetry to the PATH
# import os
# os.environ["PATH"] += ":/root/.local/bin"

# !poetry init -n
# !poetry install
# ! poetry add git+https://github.com/huggingface/transformers.git


In [ ]:
import sys
import logging
from pathlib import Path

cwd = Path.cwd()
project_root = f'{cwd.parent}'

sys.path.append(project_root)

print('The project directory is:', project_root)
print('The working directory is:', cwd)


for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
#
# self.logger = logging.getLogger(self.__class__.__name__)

logger = logging.getLogger(__name__)
# Install required packages
# !pip install markitdown
# !pip install git+https://github.com/huggingface/transformers.git triton

import os
# from markitdown import MarkItDown
from langchain.text_splitter import RecursiveCharacterTextSplitter
import torch
 from src.tokenizer import Tokenizer
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
from google.colab import userdata

HF_TOKEN = userdata.get('HUGGING_FACE_HUB_TOKEN')


LLM class:

In [ ]:
# from fuzzywuzzy import fuzz
import torch
from src.tokenizer import Tokenizer

from transformers import AutoModelForCausalLM
#     'meta-llama/Llama-3.1-8B',
class LLM:

    chat_history = []

    def __init__(self, model_name: str, token: str):
        self.model_name = model_name

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token=HF_TOKEN,
            device_map="auto" if device == "cuda" else None,
            torch_dtype=torch.float16,
            max_memory={0: "38GiB"}  # if 40GB available, leave headroom
        ).to(device)
        self.model.eval()
        self.tokenizer = Tokenizer(self.model_name, token=HF_TOKEN)

    def add_user_input(self, text: str):
        self.chat_history.append({'role': 'user', 'content': text})

    def add_model_output(self, text: str):
        self.chat_history.append({'role': 'assistant', "content": text})

    SYSTEM = "<|start_header_id|>system<|end_header_id|>"
    USER = "<|start_header_id|>user<|end_header_id|>"
    ASSISTANT = "<|start_header_id|>assistant<|end_header_id|>"
    EOT = "<|eot_id|>"
    BEGIN = "<|begin_of_text|>"

    def format_chat(self, system_msg, messages):
        prompt = self.BEGIN + self.SYSTEM + "\n" + system_msg.strip() + self.EOT
        for msg in messages:
            role = msg["role"]
            content = msg["content"].strip()
            if role not in ["user", "assistant"]:
                raise ValueError(f"Invalid role: {role}")
            prompt += f"{'<|start_header_id|>'}{role}{'<|end_header_id|>'}\n{content}{self.EOT}"
        return prompt


    def run_prompt(self, context, question, audience):

        full_prompt = f"""You are a legal assistant analyzing ICTR regulations.
          Answer the question CONCISELY using ONLY the provided document excerpt.
          You MUST include specific requirements when mentioned.
          Do NOT generate any additional information or options.
          Document Excerpt:
          {context}
          Question: {question}
          Answer in this exact format:
            "According to the document, [direct answer citing specific requirements]."

          Direct Answer:"""

        # self.tokenizer(full_prompt, return_tensors='pt', truncation=True, max_length=1024)
        inputs = self.tokenizer.tokenize(full_prompt, max_length=1024)
        input_length = inputs['input_ids'].shape[1]

        # for turn in self.chat_history:
        #       role = "User" if turn["role"] == "user" else "Assistant"
        #       conversation += f"{role}: {turn['content'].strip()}\n"

        # conversation += f"User: {current_question.strip()}\nAssistant:"

        # full_prompt = base_prompt + conversation


        if torch.cuda.is_available():
            logger.info('Using GPU.')
            self.model.to('cuda')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}

        # TODO - increase the token limit to allow for more text
        eos_token_id = self.tokenizer.eos_token_id or self.tokenizer.convert_tokens_to_ids('<|end_of_text|>')

        generation_config = {
          'max_new_tokens': 500,
          'eos_token_id': eos_token_id,
          'no_repeat_ngram_size': 3,
          'repetition_penalty': 1.2,
          'pad_token_id': eos_token_id,
          'do_sample': False, # ensures the model stays strictly factual and consistent with the source material plus it always returns the consistency of the reponse.
          # 'num_beams': 3,  # Small beam width for better answers
          # temperature=0.3,
          # do_sample=True,

        }

        with torch.no_grad():
            outputs = self.model.generate(
              **inputs,
              **generation_config
            )

        # strip out the context and question before decoding
        new_tokens = outputs[0, input_length:]
        answer = self.tokenizer.decode(new_tokens).strip()

        response = self.tokenizer.decode(outputs)
        return response.split("Answer:")[-1].strip()



llm = LLM('meta-llama/Llama-3.1-8B', token=HF_TOKEN)

torch.cuda.empty_cache()

context =  "Translation. Pursuant to Regulation 39(1) of the Regulations of the Court, all documents and materials filed with the Registry shall be in a working language of the Court. If segments of the video are not in a working language of the Court, those segments must be translated into a working language of the Court before they can be deemed admissible."

print(llm.run_prompt(context, 'what language must videos be in to be admissable?'))

In [ ]:
# vecorisation could be improved

class DocumentProcessor:
    def __init__(self):
        """Initialize the document processor with necessary components."""
        # Set up embedding model
        self.tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        self.model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        self.model.eval()
        # Initialize document converter
        self.md = MarkItDown()
        # Set up vector database

    def process_document(self, file_path):
        """Convert document to text and generate embeddings."""
        try:
            pdf_processor = PDFProccessor(file_path)
            # Convert document to text
            conversion_result = self.md.convert(file_path)
            conversion_result_text = self.md.convert(file_path).text_content
            # TODO - teh leiden guidelines contain a section of keywwords for each section - these should be parsed out and each section should be stored seperately
            conversion_result_text = pdf_processor.remove_page_numbers(conversion_result_text)
            print(conversion_result)

            # Create embeddings
            inputs = self.tokenizer(
                conversion_result_text,
                return_tensors='pt',
                truncation=True
            )
            # Use GPU if available
            if torch.cuda.is_available():
                self.model.to('cuda')
                inputs = {k: v.to('cuda') for k, v in inputs.items()}
            # Generate embeddings
            with torch.no_grad():
                outputs = self.model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().tolist()
            # **************
            return {
                'text': conversion_result_text,
                'embeddings': embeddings,
                'metadata': {}
            }
            # Note: we don't seem to get metadata from the docs anyway so better manually adding
            getattr(conversion_result, 'metadata', {})
        except Exception as e:
            logger.error(f'Error processing document {file_path}: {str(e)}')
            raise

# Initialize processor
processor = DocumentProcessor()
# Move to GPU if available
if torch.cuda.is_available():
    processor.model = processor.model.to('cuda')

Process Documents

This cell processes all documents in your legal_documents folder:

In [ ]:
# Get list of documents
# improves the inteface
# disclaimers - agent answers - next agent makes it accesible - next agent is a lawyer that critiques answer

document_files = [
    f for f in os.listdir(DOCUMENTS_PATH)
    if f.endswith(('.pdf', '.docx', '.txt', '.html', '.pptx'))
]
if not document_files:
    print('⚠️ No documents found! Add some to your legal_documents folder')
else:
    print(f'Found {len(document_files)} documents to process')
    for idx, document in enumerate(document_files):
        print(f'Processing {document}...')
        file_path = os.path.join(DOCUMENTS_PATH, document)
        # Process document
        result = processor.process_document(file_path)
        # Store in database
        doc_id = f'doc_{idx}_{document}'
        metadata = metadata_list[doc_id]
        processor.store_document(
            doc_id=doc_id,
            text=result['text'],
            embedding=result['embeddings'],
            metadata=metadata
        )
        print(f'✅ Finished storing {document} in Chroma\n')
        # except Exception as e:
        #     print(f"Error processing {document}: {str(e)}")

Question-Answering Function

This cell defines the function that generates answers using LLaMA. You may alter the values if you know what you’re doing :)

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')
# Sample function for extracting keywords
def extract_keywords(text):
    doc = nlp(text)
    keywords = [token.text for token in doc if token.is_alpha and not token.is_stop]
    return keywords


In [ ]:
!pip install fuzzywuzzy

In [ ]:
# --- Document Excerpts ---
# {truncated_context}
# Function to enhance the query by adding the extracted keywords
from fuzzywuzzy import fuzz

titles_list = [doc_metadata['title'] for doc_metadata in metadata_list.values()]

def find_full_title(query, titles_list, threshold=60):
    matches = [title for title in titles_list if fuzz.partial_ratio(query.lower(), title.lower()) >= threshold]
    return matches

keywords_list = [doc_metadata['keywords'] for doc_metadata in metadata_list.values()]

def find_keywords(search_terms, keywords):
    for search_term in search_terms:
        if search_term.lower() in keywords:
            return search_term.lower()
    return None  # No matching title found

# TODO - if the chat bot doesn't recognise any keywords it should prompt back and say something like:
#  'Ask me about digital evidence related to photographic, video or etc. evidence.'
def ask_question_llama(question, leiden_guide_lines, case_law):
    # always Leiden Guidelines
    metadata_titles = [titles_list[0]]
    if case_law:
      # Extrapolations from case law
      metadata_titles.append(titles_list[1])
    # if
      # Cases from the ICC, ICTR, ICTY, IRMCT, SCSL and STL
    # if
      # UN HUMAN RIGHTS FACT-FINDING
    # if
      # INTERNATIONAL CRIMINAL LAW
    extracted_keywords = extract_keywords(question)

    metadata_keywords = find_keywords(extracted_keywords, keywords_list)
    # metadata_titles = find_full_title(question, titles_list)
    """Generate an answer to a legal question using LLaMA."""
    # Get relevant documents
    relevant_docs = processor.find_relevant_documents(query=question, metadata_keywords=metadata_keywords, metadata_titles=metadata_titles, n_results=5)

    # Prepare context
    context_pieces = [doc['text'][4000:5000] for doc in relevant_docs]
    titles = [doc['metadata']['title'] for doc in relevant_docs]
    keywords = [doc['metadata']['keywords'] for doc in relevant_docs]
    truncated_context = '\n'.join(context_pieces)
    # Create prompt

    full_prompt = f"""You are a Human Rights Lawyer using the documents below to answer the following question.
--- Question ---
{question}

Based on the documents above, provide a clear, concise answer. If relevant, refer to legal precedent, case law, or any specific details from the documents. Do not simply restate the question; make sure the answer is grounded in the provided content.


Answer:
"""

    # Prepare for generation
    if torch.cuda.is_available():
        model.to('cuda')
    # TODO - increase the token limit to allow for more text
    inputs = tokenizer(
        full_prompt,
        return_tensors='pt',
        max_length=1024,
        truncation=True
    )
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    # TODO - increase the token limit to allow for more text
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.7,
            do_sample=True
        )
    # Process output
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'Answer:' in raw_output:
        final_answer = raw_output.split('Answer:', 1)[1].strip()
    else:
        final_answer = raw_output

    docs = ',\n - '.join(metadata_titles)
    final_answer = final_answer + f'\n\n DOCUMENTS: {docs}' + '\n\n NOTE: this is only guidance based on past case law.'
    # print(final_answer )
    return final_answer

    #

answer = ask_question_llama('Are you using the Leiden Guidelines and case law ?', True, True)
print(answer)

Create User Interface

Finally, we can also create the Gradio interface:

In [ ]:
# !pip install gradio
import gradio as gr
def ask_question_llama():
  pass
def create_interface():
    demo = gr.Interface(
        fn=ask_question_llama,
        inputs=[
            gr.Dropdown(
            ["Lawyer", "Open Source Researcher", "Journalist"], label="Role", info="How would you describe your role?"),
             gr.Dropdown(
            ['Videos',
             'Photographs',
             'Aerial and Satellite Images',
             'Intercepts',
             'Call Data Records',
             'Audio Recordings'],
            label="Evidence",
            info="What form of Digitally Derived Evidence are you interested in?"),
            gr.Textbox(
              label='Your Question',
              placeholder='Ask any question about Digitally Derived Evidence...',
              lines=3
          ),

            ],

        outputs=[
            gr.Markdown(
                label='Answer',
            )
        ],

        title='Digitally Derived Evidence',
        description='This AI assistant can answer questions Digitally Derived Evidence and particularly the Leiden Guidelines.'
    )
    return demo
# Launch interface
demo = create_interface()
demo.launch(share=True)